# 03 · Estado in-game por checkpoint y dataset final

A partir de los tiros de FotMob se calcula el estado de cada partido en los minutos 15, 30, 45, 60 y 75; luego se agrega el contexto previo de D2.

**Requiere:** haber ejecutado `02_union_partidos.ipynb` (usa `mapa_d1_d2.csv`)
**Lee:** `data/d1/matches/*.json` (sección `shotmap`), `data/d2/matches.csv`, `data/d2/match_prediction_features.csv`
**Genera:** `data/processed/estado_d1.csv` y `data/processed/instances.csv` (520 instancias)

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

D1 = Path("../data/d1/matches")
D2 = Path("../data/d2")
PROCESADOS = Path("../data/processed")
CHECKPOINTS = [15, 30, 45, 60, 75]

def tiros_de_partido(ruta):
    d = json.load(open(ruta, encoding="utf-8"))
    g = d["general"]
    shots = (d["content"].get("shotmap") or {}).get("shots", [])
    filas = []
    for s in shots:
        filas.append({
            "fotmob_id": int(g["matchId"]),
            "home_id": g["homeTeam"]["id"], "home": g["homeTeam"]["name"],
            "away_id": g["awayTeam"]["id"], "away": g["awayTeam"]["name"],
            "team_id": s.get("teamId"),
            "min": s.get("min"), "min_added": s.get("minAdded"), "period": s.get("period"),
            "evento": s.get("eventType"), "xg": s.get("expectedGoals"),
            "al_arco": s.get("isOnTarget"), "autogol": s.get("isOwnGoal"),
        })
    return pd.DataFrame(filas)

tiros = pd.concat([tiros_de_partido(f) for f in sorted(D1.glob("*.json"))], ignore_index=True)
print(tiros.shape)
print(tiros.fotmob_id.nunique(), "partidos con tiros")
print(tiros.period.value_counts())
print(tiros.evento.value_counts())

(2630, 13)
104 partidos con tiros
period
SecondHalf         1368
FirstHalf          1137
SecondHalfExtra      44
FirstHalfExtra       41
PenaltyShootout      40
Name: count, dtype: int64
evento
AttemptSaved    1253
Miss             994
Goal             333
Post              50
Name: count, dtype: int64


## 1. Verificaciones de los tiros

- El tiempo añadido viene dentro de `FirstHalf` y `SecondHalf`; la prórroga y la tanda de penales tienen periodos propios.
- Los tiros sin xG deberían ser los autogoles.
- Sin la tanda de penales deberían quedar 308 goles, igual que en D2.

In [2]:
print(tiros.groupby("period")["min"].agg(["min", "max"]))
print()
print("tiros sin xG:", tiros.xg.isna().sum())
print(tiros[tiros.xg.isna()][["evento", "autogol", "period"]].value_counts())
print()
goles = tiros[(tiros.evento == "Goal") & (tiros.period != "PenaltyShootout")].copy()
print("goles sin tanda:", len(goles), "| autogoles:", goles.autogol.sum())

                 min  max
period                   
FirstHalf          1   45
FirstHalfExtra    91  105
PenaltyShootout  121  132
SecondHalf        46   90
SecondHalfExtra  106  120

tiros sin xG: 14
evento  autogol  period         
Goal    True     SecondHalf         8
                 FirstHalf          5
                 SecondHalfExtra    1
Name: count, dtype: int64

goles sin tanda: 308 | autogoles: 14


## 2. Tiros, tiros al arco y xG acumulados en cada checkpoint

Solo cuentan el 1.er y el 2.º tiempo. El checkpoint 45 incluye el tiempo añadido del primer tiempo. Los autogoles no cuentan como tiro del equipo.

In [3]:
t90 = tiros[tiros.period.isin(["FirstHalf", "SecondHalf"])].copy()
t90["xg"] = t90["xg"].fillna(0)
t90["lado"] = np.where(t90.team_id == t90.home_id, "A", "B")

def hasta(df, t):
    """Eventos ocurridos hasta el checkpoint t."""
    if t < 45:
        return df[(df.period == "FirstHalf") & (df["min"] <= t)]
    if t == 45:
        return df[df.period == "FirstHalf"]          # incluye el 45+x
    return df[(df.period == "FirstHalf") |
              ((df.period == "SecondHalf") & (df["min"] <= t))]

filas = []
for mid, g in t90.groupby("fotmob_id"):
    for t in CHECKPOINTS:
        h = hasta(g, t)
        intentos = h[~h.autogol.astype(bool)]   # un autogol no es un tiro del equipo
        fila = {"fotmob_id": mid, "t": t}
        for lado in ["A", "B"]:
            s = intentos[intentos.lado == lado]
            fila[f"xg_{lado}"] = s.xg.sum()
            fila[f"tiros_{lado}"] = len(s)
            fila[f"al_arco_{lado}"] = int(s.al_arco.sum())
        filas.append(fila)

estado = pd.DataFrame(filas)
print(estado.shape)          # esperamos (520, 8)
estado.groupby("t")[["xg_A", "xg_B", "tiros_A", "tiros_B"]].mean().round(2)

(520, 8)


,xg_A,xg_B,tiros_A,tiros_B
t,,,,
15,0.20,0.15,1.87,1.39
30,0.34,0.30,3.22,2.54
45,0.64,0.50,6.16,4.72
60,0.90,0.65,8.29,6.42
75,1.11,0.79,10.08,7.71


## 3. Goles por checkpoint y etiqueta

Los autogoles vienen registrados con el equipo que los comete. Se prueban las dos reglas posibles contra el marcador oficial de D2; la correcta es sumárselos al rival.

In [4]:
mapa = pd.read_csv(PROCESADOS / "mapa_d1_d2.csv")
oficial = pd.read_csv(D2 / "matches.csv")[["match_id", "home_score", "away_score"]]
ids = tiros.fotmob_id.unique()      # los 104 partidos, incluidos los 0-0

def lado_gol(goles, autogol_al_rival):
    lado = np.where(goles.team_id == goles.home_id, "A", "B")
    if autogol_al_rival:
        lado = np.where(goles.autogol.astype(bool), np.where(lado == "A", "B", "A"), lado)
    return lado

for regla in [True, False]:
    g = goles.assign(lado=lado_gol(goles, regla))
    tot = (g.pivot_table(index="fotmob_id", columns="lado", aggfunc="size", fill_value=0)
             .reindex(ids, fill_value=0).rename_axis("fotmob_id").reset_index())
    cmp = tot.merge(mapa, on="fotmob_id").merge(oficial, on="match_id")
    ok = ((cmp.A == cmp.home_score) & (cmp.B == cmp.away_score)).sum()
    print("autogol se suma al rival:" if regla else "autogol se suma al equipo registrado:", ok, "/ 104")

autogol se suma al rival: 104 / 104
autogol se suma al equipo registrado: 90 / 104


In [5]:
goles["lado"] = lado_gol(goles, autogol_al_rival=True)
g90 = goles[goles.period.isin(["FirstHalf", "SecondHalf"])]   # sin prórroga

# goles acumulados en cada checkpoint
filas = []
for mid in ids:
    g = g90[g90.fotmob_id == mid]
    for t in CHECKPOINTS:
        h = hasta(g, t)
        filas.append({"fotmob_id": mid, "t": t,
                      "goles_A": int((h.lado == "A").sum()),
                      "goles_B": int((h.lado == "B").sum())})
goles_t = pd.DataFrame(filas)

# etiqueta: resultado a los 90'  (0 = gana A, 1 = empate, 2 = gana B)
fin = (g90.groupby(["fotmob_id", "lado"]).size().unstack(fill_value=0)
          .reindex(columns=["A", "B"], fill_value=0)
          .reindex(ids, fill_value=0).rename_axis("fotmob_id"))
dif = fin["A"] - fin["B"]
etiqueta = pd.DataFrame({"fotmob_id": fin.index,
                         "y": np.select([dif > 0, dif == 0], [0, 1], 2)})

estado = estado.merge(goles_t, on=["fotmob_id", "t"]).merge(etiqueta, on="fotmob_id")
print(estado.shape)    # esperamos (520, 11)
print(estado.drop_duplicates("fotmob_id").y.map({0: "gana A", 1: "empate", 2: "gana B"}).value_counts())
print("¿goles que bajan entre checkpoints?:",
      (estado.groupby("fotmob_id")[["goles_A", "goles_B"]].diff() < 0).any().any())

estado.to_csv(PROCESADOS / "estado_d1.csv", index=False)

(520, 11)
y
gana A    46
empate    29
gana B    29
Name: count, dtype: int64
¿goles que bajan entre checkpoints?: False


## 4. Contexto previo y dataset final

Todas las variables se expresan como "A comparado con B". Elo en centenas; ranking y valor del plantel en logaritmo, porque abarcan varios órdenes de magnitud. El ranking se resta al revés (B − A) para que un valor positivo signifique que A está mejor rankeado.

In [6]:
X2 = pd.read_csv(D2 / "match_prediction_features.csv")

ctx = pd.DataFrame({
    "match_id":    X2.match_id,
    "elo_diff":    (X2.home_elo - X2.away_elo) / 100,
    "rank_diff":   np.log(X2.away_fifa_rank) - np.log(X2.home_fifa_rank),
    "value_diff":  np.log(X2.home_squad_total_value_eur) - np.log(X2.away_squad_total_value_eur),
    "host_diff":   X2.home_is_host - X2.away_is_host,
    "rest_diff":   X2.home_rest_days - X2.away_rest_days,
    "is_knockout": X2.is_knockout,
})

inst = estado.merge(mapa, on="fotmob_id").merge(ctx, on="match_id")
assert not inst.invertido.any()     # si un partido tuviera el local invertido, habría que intercambiar A y B

inst["goal_diff"]    = inst.goles_A - inst.goles_B
inst["gd_x_t"]       = inst.goal_diff * inst.t / 90
inst["xg_diff"]      = inst.xg_A - inst.xg_B
inst["tiros_diff"]   = inst.tiros_A - inst.tiros_B
inst["al_arco_diff"] = inst.al_arco_A - inst.al_arco_B

assert inst.shape[0] == 520 and inst.isna().sum().sum() == 0
print(inst.shape, "| valores faltantes:", inst.isna().sum().sum())
inst[["t", "goal_diff", "xg_diff", "tiros_diff", "elo_diff", "value_diff", "y"]].describe().round(2)

(520, 24) | valores faltantes: 0


,t,goal_diff,xg_diff,tiros_diff,elo_diff,value_diff,y
count,520.00,520.00,520.00,520.00,520.00,520.00,520.00
mean,45.00,0.12,0.16,1.37,0.57,0.34,0.84
std,21.23,1.18,0.86,5.64,2.36,1.37,0.83
min,15.00,-4.00,-3.04,-25.00,-5.40,-2.82,0.00
25%,30.00,-1.00,-0.18,-2.00,-1.12,-0.79,0.00
50%,45.00,0.00,0.12,1.00,0.60,0.54,1.00
75%,60.00,1.00,0.63,4.00,2.26,1.22,2.00
max,75.00,5.00,3.14,19.00,5.10,3.21,2.00


In [7]:
inst.to_csv(PROCESADOS / "instances.csv", index=False)
print("Guardado:", PROCESADOS / "instances.csv")

Guardado: ../data/processed/instances.csv
